# 22 Transformer 与 GPT

依赖安装说明：`pip install numpy matplotlib scikit-learn torch`

这个 notebook 是系列里的 Transformer/GPT 入门版。更完整的纯 Python 拆解可以继续看 `microgpt_explained_zh.ipynb`。

GPT 的核心任务是：根据前面的 token，预测下一个 token。


## 1. 数学逻辑：自注意力

给定输入矩阵 `X`，先投影成 Q、K、V：

$$Q=XW_Q,\quad K=XW_K,\quad V=XW_V$$

注意力分数：

$$S=\frac{QK^T}{\sqrt{d_k}}$$

注意力权重：

$$A=softmax(S)$$

输出：

$$O=AV$$

GPT 使用 causal mask，当前位置只能看自己和过去，不能偷看未来。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def softmax_np(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

# 从零演示 scaled dot-product attention
T, d = 5, 4
X = np.random.normal(size=(T, d))
Wq = np.random.normal(size=(d, d))
Wk = np.random.normal(size=(d, d))
Wv = np.random.normal(size=(d, d))
Q, K, V = X @ Wq, X @ Wk, X @ Wv
scores = Q @ K.T / np.sqrt(d)
mask = np.triu(np.ones((T, T)), k=1).astype(bool)
scores[mask] = -1e9
weights = softmax_np(scores, axis=1)
out = weights @ V

print('attention weights shape:', weights.shape)
plt.imshow(weights, cmap='viridis')
plt.title('causal attention：右上角未来位置被 mask')
plt.xlabel('key position')
plt.ylabel('query position')
plt.colorbar()
plt.show()


## 2. GPT 训练目标

字符级 GPT 会把文本变成 token 序列：

$$[x_0, x_1, x_2, \cdots, x_T]$$

训练时每个位置预测下一个 token：

$$p(x_{t+1}|x_0,\cdots,x_t)$$

交叉熵损失：

$$L=-\frac{1}{T}\sum_t \log p(x_{t+1}|x_{\le t})$$

生成时，从开始 token 出发，循环抽样下一个 token。


In [ ]:
# PyTorch 实战：极小字符级 GPT 骨架，不追求效果，只看结构
import torch
from torch import nn
import torch.nn.functional as F

text = 'hello transformer. hello gpt. tiny model for learning.'
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

block_size = 8
vocab_size = len(chars)
n_embd = 24

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.attn = nn.MultiheadAttention(n_embd, num_heads=4, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.ReLU(), nn.Linear(4*n_embd, n_embd))
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)[None, :, :]
        mask = torch.triu(torch.ones(T, T, device=idx.device), diagonal=1).bool()
        attn_out, _ = self.attn(x, x, x, attn_mask=mask)
        x = x + attn_out
        x = x + self.mlp(x)
        return self.lm_head(x)

model = TinyGPT()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

def get_batch(batch_size=16):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

for step in range(120):
    xb, yb = get_batch()
    logits = model(xb)
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print('训练后的最后 loss:', round(loss.item(), 3))


In [ ]:
# 生成文本：每次只取最后 block_size 个 token 作为上下文
@torch.no_grad()
def generate(start='h', max_new_tokens=80, temperature=0.9):
    idx = torch.tensor([[stoi[start]]], dtype=torch.long)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits = model(idx_cond)[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    return ''.join(itos[int(i)] for i in idx[0])

print(generate('h'))


## 3. 常见误区

- Transformer 不等于 GPT；GPT 是 decoder-only Transformer 的一种语言模型用法。
- attention 权重可以帮助理解模型关注哪里，但不能完整解释模型推理。
- GPT 训练需要大量数据；这个小例子只用于看结构，不代表真实效果。

## 4. 小实验

- 改 `block_size`，观察上下文长度。
- 改 `n_embd` 和 `num_heads`。
- 改 `temperature`，观察生成随机性。
- 对照 `microgpt_explained_zh.ipynb`，理解纯 Python 版如何自己实现 autograd。
